# 01｜TensorFlow：資料 → 模型 → `fit`

對照原 `ML&DL_筆記.md` 的 `tf.data`、Dense、Sequential、交叉熵與 Keras 訓練。離線合成資料只用來練 API，分數不是業務結果。完整配對腳本見 `../../examples/tf_classification.py` 與 PyTorch 版 `../../examples/torch_classification_pair.py`。

In [ ]:
import numpy as np
import tensorflow as tf

tf.keras.utils.set_random_seed(42)
rng = np.random.default_rng(42)
x = rng.normal(size=(800, 2)).astype('float32')
y = (x[:, 0] + 0.7 * x[:, 1] > 0).astype('int32')
order = rng.permutation(len(x))
train_idx, valid_idx = order[:640], order[640:]
x_train, y_train = x[train_idx], y[train_idx]
x_valid, y_valid = x[valid_idx], y[valid_idx]
print(x_train.shape, y_train.shape, np.bincount(y_train), np.bincount(y_valid))

資料切分應先依真實產生過程決定：時間資料按時間，重複個體按群組；這裡的隨機切分只適合獨立樣本。`from_tensor_slices` 包住已在記憶體中的陣列，不能用它聲稱串流資料永不佔 RAM。

In [ ]:
train = (tf.data.Dataset.from_tensor_slices((x_train, y_train))
         .shuffle(len(x_train), seed=42).batch(64).prefetch(tf.data.AUTOTUNE))
valid = tf.data.Dataset.from_tensor_slices((x_valid, y_valid)).batch(64)
xb, yb = next(iter(train))
print(xb.shape, xb.dtype, yb.shape, yb.dtype)

In [ ]:
model = tf.keras.Sequential([
    tf.keras.Input(shape=(2,)),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(2),  # 二類 logits，不先 softmax
])
model.compile(optimizer=tf.keras.optimizers.AdamW(learning_rate=0.01, weight_decay=0.01),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name='accuracy')])
model.summary()
history = model.fit(train, validation_data=valid, epochs=8, shuffle=False, verbose=0)
print('last val_accuracy:', history.history['val_accuracy'][-1])

In [ ]:
val_loss, val_accuracy = model.evaluate(valid, verbose=0)
logits = model(x_valid[:3], training=False)
probabilities = tf.nn.softmax(logits, axis=-1)
print('validation:', round(val_loss, 4), round(val_accuracy, 3))
print('logits shape:', logits.shape, 'probabilities:', probabilities.numpy().round(3))
assert val_accuracy > 0.9
assert np.allclose(probabilities.numpy().sum(axis=1), 1.0)

下一本 `02_gradient_callbacks_save.ipynb` 拆開 `fit` 所做的前向傳播、loss、反向傳播及最佳模型保存。不同種子／框架的分數不能直接拿來宣稱框架優劣。